# MyWeather – Coordinate-Based Weather Lookup Application

This notebook is an interactive technical walkthrough of **MyWeather**. The application accepts latitude and longitude copied from Google Maps, then uses standard OpenWeather endpoints to display the resolved location, current weather, an interactive weather map, air quality, and a five-day forecast.

> Security note: this notebook contains no real API key. All demonstration payloads are sanitized local samples.

## 1. Project objectives

- Provide a lightweight coordinate-based weather dashboard with Streamlit.
- Validate coordinates: latitude must be in `[-90, 90]` and longitude in `[-180, 180]`.
- Resolve coordinates to a city, an optional state, and an ISO 3166-1 alpha-2 country code.
- Present current weather, an interactive map, and air-quality information.
- Add interactive forecast charts, local user accounts, per-user saved locations, and tests that do not consume real API calls.

## 2. Requirements analysis

The application requirements are divided into core functional requirements, extended functional requirements, and cross-cutting quality requirements. Core requirements define the minimum weather lookup workflow. Extended requirements improve usability and persistence, while quality requirements address security, reliability, and maintainability.

### Core capabilities

1. Accept and validate coordinates; perform reverse geocoding with `limit=1`.
2. Retrieve temperature, feels-like temperature, humidity, wind speed, a Vietnamese weather description, and an icon.
3. Display a Folium map with `temp_new`, `wind_new`, `clouds_new`, and `precipitation_new` overlays.
4. Display OpenWeather AQI values from 1 to 5 and concentrations for PM2.5, PM10, CO, NO₂, O₃, and SO₂, plus NO and NH₃ when available.

### Additional capabilities

- A custom summary widget rendered through `st.iframe`. The official widget is not embedded because this application uses a coordinate-first workflow, and embedding an additional third-party script is unnecessary.
- Five-day/three-hour forecast charts for temperature, humidity, and wind speed using Plotly.
- Independent service calls allow partial results to remain visible when one OpenWeather endpoint is unavailable.
- Optional local accounts use Argon2 password hashes and SQLite-backed saved locations with ownership isolation.

### Quality requirements

- Keep API calls separate from Streamlit presentation code.
- Read credentials from secrets or environment variables and never hardcode a real API key.
- Use bounded HTTP timeouts, safe error messages, and TTL-based caching.
- Handle missing optional API fields without claiming unavailable data exists.
- Test network behavior with mocks so automated tests do not consume API quota.

## 3. Technology stack

| Component | Purpose |
|---|---|
| Python 3.11+ | Implementation language |
| Streamlit | User interface and result caching |
| requests | HTTP requests with a bounded timeout |
| Folium + streamlit-folium | Interactive map and tile overlays |
| pandas + Plotly | Forecast data preparation and charts |
| SQLite | Local account and saved-location persistence |
| argon2-cffi | Salted Argon2id password hashing |
| pytest | Unit testing with mocked HTTP responses |

## 4. Overall architecture

```mermaid
flowchart LR
    U[User enters latitude and longitude] --> V[core.validation]
    V --> UI[app.py and Streamlit cache]
    UI --> G[services.geocoding_service]
    UI --> W[services.weather_service]
    UI --> A[services.air_quality_service]
    UI --> F[services.forecast_service]
    G & W & A & F --> C[services.api_client.request_json]
    C --> O[OpenWeather API]
    UI --> D[views.dashboard]
    UI --> M[views.weather_map and Folium]
    UI --> P[Plotly charts]
    UI --> AU[services.auth_service]
    UI --> SL[services.saved_location_service]
    AU & SL --> DB[(SQLite database)]
```

`services/api_client.py` centralizes timeouts and converts HTTP status codes into safe user-facing messages. Each service parses data for one domain, `views/` contains presentation code, and `app.py` coordinates the workflow. This separation allows core behavior to be tested without starting Streamlit.

In [ ]:
# Prepare the import path whether this notebook is opened from the project root
# or from the notebooks/ directory.
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

## 5. OpenWeather API-key setup

1. Create an account at [OpenWeather](https://home.openweathermap.org/users/sign_up) and verify the email address.
2. Open the **API keys** page and copy the default key or create a new one. A newly created key may require some time before activation.
3. Choose one of the following configuration methods. Never commit the file that contains the real key.

```toml
# .streamlit/secrets.toml
OPENWEATHER_API_KEY = "your_real_key_here"
```

Alternatively, set the `OPENWEATHER_API_KEY` environment variable in the shell. The repository provides only `.streamlit/secrets.toml.example` to avoid maintaining duplicate secret files; the real secrets file is excluded by `.gitignore`.

## 6. API integration

| Purpose | Endpoint | Main parameters | Used response fields |
|---|---|---|---|
| Reverse geocoding | `/geo/1.0/reverse` | `lat`, `lon`, `limit=1`, `appid` | `name`, `state`, `country` |
| Current weather | `/data/2.5/weather` | `lat`, `lon`, `appid`, `units=metric`, `lang=vi` | `main`, `wind`, `weather` |
| Air pollution | `/data/2.5/air_pollution` | `lat`, `lon`, `appid` | `main.aqi`, `components` |
| Forecast | `/data/2.5/forecast` | `lat`, `lon`, `appid`, `units=metric` | `list`, `city.timezone` |
| Map tiles | `/map/{layer}/{z}/{x}/{y}.png` | `appid` | Raster PNG image |

The displayed AQI uses the **OpenWeather 1–5 scale**: Good, Fair, Moderate, Poor, and Very Poor. It is not the US EPA AQI. Pollutant concentrations are returned in µg/m³.

## 7. Server-side processing design

The server-side code separates HTTP transport, domain parsing, validation, and presentation. This keeps OpenWeather calls testable without starting Streamlit and prevents raw provider responses or credentials from being exposed to the UI.

| Module / function | Responsibility | Input | Processed output |
|---|---|---|---|
| `core.validation.validate_coordinates` | Validate user coordinates before network access | String or numeric latitude/longitude | Bounded finite `float` pair or a validation error |
| `services.api_client.request_json` | Perform all OpenWeather HTTP requests | Endpoint, parameter mapping, timeout | Parsed JSON or a credential-safe `WeatherAPIError` |
| `geocoding_service.reverse_geocode` | Call reverse geocoding with `limit=1` | Coordinates and API key | `LocationInfo` or `None` for an empty result |
| `weather_service.get_current_weather` | Request metric current weather with `lang=vi` | Coordinates and API key | `CurrentWeather` with optional fields represented as `None` |
| `air_quality_service.get_air_quality` | Request AQI and pollutant concentrations | Coordinates and API key | `AirQuality` with the OpenWeather 1–5 label |
| `forecast_service.get_forecast` | Request and normalize the 5-day/3-hour forecast | Coordinates and API key | Timezone-aware `ForecastData` points |
| `parse_*` functions | Convert sanitized JSON dictionaries into typed dataclasses | API response payload | UI-ready domain objects |
| Cached wrappers in `app.py` | Reduce repeated requests | Coordinates and API key | Cached service results with 10- or 30-minute TTL |

`request_json` uses a 10-second timeout and converts connection failures, invalid JSON, HTTP 401, 404, 429, unexpected statuses, and 5xx responses into fixed messages that do not include the request URL or API key. Each API is fetched independently, so one unavailable feature does not hide other successful results. The parsers use defensive dictionary access because OpenWeather can omit optional observations.

In [ ]:
from core.validation import CoordinateValidationError, validate_coordinates

valid = validate_coordinates('10.8231', '106.6297')
print('Valid coordinates:', valid)

for pair in [('abc', '106'), ('91', '0'), ('0', '181')]:
    try:
        validate_coordinates(*pair)
    except CoordinateValidationError as error:
        print(f'{pair}: {error}')

Each module exposes an independent `parse_*` function. This design allows missing fields and malformed data to be tested with sanitized JSON samples without using the network.

In [ ]:
from services.geocoding_service import parse_reverse_geocoding
from services.weather_service import parse_current_weather
from services.air_quality_service import parse_air_quality

sample_location = [{'name': 'Ho Chi Minh City', 'state': 'Ho Chi Minh City',
                    'country': 'VN', 'lat': 10.8231, 'lon': 106.6297}]
sample_weather = {
    'main': {'temp': 31.2, 'feels_like': 35.0, 'humidity': 70},
    'wind': {'speed': 3.4},
    'weather': [{'description': 'scattered clouds', 'icon': '03d'}],
    'dt': 1710000000,
}
sample_air = {'list': [{'dt': 1710000000, 'main': {'aqi': 2}, 'components': {
    'pm2_5': 12.3, 'pm10': 18.2, 'co': 220.1, 'no': 1.2,
    'no2': 8.0, 'o3': 45.0, 'so2': 3.0, 'nh3': 0.8}}]}

location = parse_reverse_geocoding(sample_location)
weather = parse_current_weather(sample_weather)
air = parse_air_quality(sample_air)
location, weather, air

## 8. Application workflow

1. A guest can paste coordinates and select **Look up weather**. A form prevents API requests on every keystroke.
2. The application validates the coordinates and reads the API key from Streamlit secrets or an environment variable.
3. Four weather services are called independently. A failure in one service does not hide valid data returned by another service.
4. A user can optionally register or sign in, save a resolved location, and load or delete only that account's locations.
5. `st.cache_data` uses a 10-minute TTL for current data and a 30-minute TTL for forecasts. There is no automatic refresh or unbounded network loop.
6. Folium renders the marker and weather tiles; Plotly creates three interactive forecast charts.

The logical API order is reverse geocoding → current weather → air pollution → forecast. The browser requests map tiles only while the map is displayed.

In [ ]:
# Build a demonstration map with a fake key. The map is not displayed here,
# so this cell does not request remote tiles.
from views.weather_map import WEATHER_LAYERS, create_weather_map

print('Weather Maps 1.0 layers confirmed from the documentation:', WEATHER_LAYERS)
demo_map = create_weather_map(10.8231, 106.6297, 'DEMO_KEY_NOT_VALID', 'Clouds')
print(type(demo_map).__name__, 'created; no network request has been sent.')

## 9. Getting started

Run the following commands from the project root:

```bash
python -m venv .venv
# Windows PowerShell: .\.venv\Scripts\Activate.ps1
# macOS/Linux: source .venv/bin/activate
python -m pip install -r requirements.txt
streamlit run app.py
```

Start this notebook with `jupyter notebook notebooks/MyWeather.ipynb` and run the tests with `python -m pytest -q`.

## 10. Basic feature testing

The test suite uses `monkeypatch`, fake API responses, and temporary SQLite databases, so it needs no API key and consumes no quota. Covered cases include coordinate validation, weather parsing, AQI mapping, forecast timezone conversion, HTTP failures, Argon2 hashing, account validation, authentication failures, duplicate saved locations, ownership isolation, and foreign-key cascades.

In [ ]:
# Demonstrate an HTTP 401 failure with a mock. No Internet request is made.
from unittest.mock import patch
from services.api_client import WeatherAPIError, request_json

class SanitizedResponse:
    status_code = 401
    def json(self):
        return {'message': 'sample provider message'}

with patch('services.api_client.requests.get', return_value=SanitizedResponse()):
    try:
        request_json('https://example.invalid', {'appid': 'NOT_A_REAL_KEY'})
    except WeatherAPIError as error:
        print(error)  # Fixed message; it contains neither the key nor request URL.

In [ ]:
# Run the complete test suite directly from the notebook.
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=project_root, capture_output=True, text=True, check=False
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## 11. Extended feature: Forecast visualization

OpenWeather timestamps are Unix values in UTC, while `city.timezone` contains an offset in seconds. `services/forecast_service.py` converts every point to a timezone-aware `datetime` before charting it. The time axis therefore represents local time using the offset supplied by the provider.

In [ ]:
from services.forecast_service import parse_forecast
import pandas as pd
import plotly.express as px

sample_forecast = {
    'city': {'timezone': 7 * 3600},
    'list': [
        {'dt': 1710000000 + step * 10800,
         'main': {'temp': 28 + step * 0.4, 'humidity': 76 - step},
         'wind': {'speed': 2.1 + step * 0.2}}
        for step in range(8)
    ],
}
parsed = parse_forecast(sample_forecast)
forecast_frame = pd.DataFrame({
    'Local time': [point.local_time for point in parsed.points],
    'Temperature (°C)': [point.temperature for point in parsed.points],
})
px.line(
    forecast_frame, x='Local time', y='Temperature (°C)', markers=True,
    title='Sanitized sample forecast – no API request'
)

## 12. Extended feature: Accounts and SQLite persistence

Accounts are optional: guests retain access to every weather feature. Registration stores a case-insensitive unique username and an Argon2id hash, never a plaintext password. Every saved-location query includes `user_id`, and deletion uses both the row ID and owner ID. SQLite foreign keys are enabled for each connection. The following example uses a temporary database and does not modify `data/myweather.db`.

In [ ]:
from tempfile import TemporaryDirectory
from data.database import connect_database, initialize_database
from services.auth_service import authenticate_user, register_user
from services.saved_location_service import list_saved_locations, save_location

with TemporaryDirectory() as temporary_directory:
    demo_database = Path(temporary_directory) / 'myweather_demo.db'
    initialize_database(demo_database)
    demo_user = register_user(
        'demo_user', 'Forecast123', 'Forecast123', demo_database
    )
    authenticated = authenticate_user('DEMO_USER', 'Forecast123', demo_database)
    save_location(
        demo_user.id, 'Ho Chi Minh City', None, 'VN',
        10.8231, 106.6297, demo_database
    )
    with connect_database(demo_database) as connection:
        stored_hash = connection.execute(
            'SELECT password_hash FROM users WHERE id = ?', (demo_user.id,)
        ).fetchone()['password_hash']
    print('Authenticated:', authenticated.username)
    print('Stored password format:', stored_hash.split('$')[1])
    print('Saved locations:', list_saved_locations(demo_user.id, demo_database))

## 13. Design notes and known limitations

The project completes the coordinate, weather, map, AQI, forecast, account, and saved-location workflows with a modular architecture, caching, request timeouts, and isolated tests. Important limitations include:

- Results depend on OpenWeather coverage, update frequency, availability, and account quota. A new key might not work immediately.
- `city.timezone` provides an offset rather than an IANA timezone identifier, so detailed daylight-saving-time rules cannot be inferred.
- Weather Maps 1.0 requires `appid` in the tile URL. Folium requests tiles in the browser, which means a user can see the key in browser network tools. The application discloses that risk. A production system should use a dedicated rotatable key and consider a server-side tile proxy with caching and rate limiting, subject to the provider's license and terms.
- The widget is a custom summary rather than the official OpenWeather widget.
- The project never assumes that a live API request succeeded. Live verification occurs only after the user configures a real key.
- Authentication is stored in Streamlit session state and does not provide persistent cookies, email verification, password recovery, or OAuth.
- Account data is always stored in the local `data/myweather.db` file. Deployments with ephemeral filesystems can erase this file during restart or redeployment.